## Homework 4: Model Building and Tracking

In this assignment, I build a machine learning model to predict finishing position in Formula 1 races and track each run using MLflow.

The target variable is `positionOrder`, which represents the driver's final finishing position in a race.

I use the following features:

- `grid`: starting position
- `laps`: number of laps completed
- `year`: race year
- `pitstop_count`: number of pit stops in the race

Since pit stop data is only available from 2011 onward, I restrict the modeling dataset to races from 2011 to 2024 so that all selected features are valid and observed.

In [0]:
import os
import tempfile

import mlflow
import mlflow.sklearn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pyspark.sql import functions as F

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## Data Loading and Cleaning

I first load the relevant datasets and cast key columns into numeric types so they can be used for joins, aggregations, and machine learning.

In [0]:
pitstops_raw_df = spark.read.csv("/Volumes/gr5069/raw/f1_data/pit_stops.csv", header=True)
results_raw_df  = spark.read.csv("/Volumes/gr5069/raw/f1_data/results.csv", header=True)
drivers_raw_df  = spark.read.csv("/Volumes/gr5069/raw/f1_data/drivers.csv", header=True)
races_raw_df    = spark.read.csv("/Volumes/gr5069/raw/f1_data/races.csv", header=True)

In [0]:
pitstops_df = (
    pitstops_raw_df
    .withColumn("raceId", F.col("raceId").cast("int"))
    .withColumn("driverId", F.col("driverId").cast("int"))
    .withColumn("stop", F.col("stop").cast("int"))
)

results_df = (
    results_raw_df
    .withColumn("raceId", F.col("raceId").cast("int"))
    .withColumn("driverId", F.col("driverId").cast("int"))
    .withColumn("grid", F.col("grid").cast("int"))
    .withColumn("laps", F.col("laps").cast("int"))
    .withColumn("positionOrder", F.col("positionOrder").cast("int"))
)

drivers_df = (
    drivers_raw_df
    .withColumn("driverId", F.col("driverId").cast("int"))
)

races_df = (
    races_raw_df
    .withColumn("raceId", F.col("raceId").cast("int"))
    .withColumn("year", F.col("year").cast("int"))
)

## Feature Engineering and Data Integration

To prepare the dataset for modeling, I first compute the number of pit stops made by each driver in each race. Then, I join this information with the race results and race year.

Because pit stop data is only available from 2011 to 2024, I restrict the modeling dataset to that time period. This avoids incorrectly treating missing pit stop information as zero.

In [0]:
# keep only races from 2011 onward
races_filtered_df = races_df.filter(F.col("year") >= 2011)

# compute pit stop count per driver per race
pitstop_counts_df = (
    pitstops_df
    .groupBy("raceId", "driverId")
    .agg(F.count("*").alias("pitstop_count"))
)

# build final modeling table
model_base_df = (
    results_df
    .join(races_filtered_df.select("raceId", "year"), on="raceId", how="inner")
    .join(pitstop_counts_df, on=["raceId", "driverId"], how="inner")
    .select(
        "raceId",
        "driverId",
        "grid",
        "laps",
        "year",
        "pitstop_count",
        "positionOrder"
    )
)

In [0]:
#Sanity checks
model_base_df.select(
    F.min("year").alias("min_year"),
    F.max("year").alias("max_year")
).show()

model_base_df.groupBy("year").count().orderBy("year").show()

In [0]:

#Prepare Pandas data for sklearn
feature_cols = ["grid", "laps", "year", "pitstop_count"]
target_col = "positionOrder"

model_pdf = model_base_df.select(feature_cols + [target_col]).toPandas()

X = model_pdf[feature_cols]
y = model_pdf[target_col]

## Model and MLflow Setup

I use a Random Forest Regressor because it supports tunable hyperparameters and works well for nonlinear relationships. I track each run using MLflow.

For each run, I log:

- the model hyperparameters
- the trained model
- evaluation metrics
- two artifacts:
  - a residual plot
  - a feature importance CSV

In [0]:
mlflow.set_experiment("/Users/cs4626@columbia.edu/hw3_finish_position_pitstop")

In [0]:
param_grid = [
    {"n_estimators": 50,  "max_depth": 3,  "random_state": 42},
    {"n_estimators": 50,  "max_depth": 5,  "random_state": 42},
    {"n_estimators": 50,  "max_depth": 8,  "random_state": 42},
    {"n_estimators": 100, "max_depth": 3,  "random_state": 42},
    {"n_estimators": 100, "max_depth": 5,  "random_state": 42},
    {"n_estimators": 100, "max_depth": 8,  "random_state": 42},
    {"n_estimators": 200, "max_depth": 3,  "random_state": 42},
    {"n_estimators": 200, "max_depth": 5,  "random_state": 42},
    {"n_estimators": 200, "max_depth": 8,  "random_state": 42},
    {"n_estimators": 300, "max_depth": 3,  "random_state": 42},
    {"n_estimators": 300, "max_depth": 5,  "random_state": 42},
    {"n_estimators": 300, "max_depth": 8,  "random_state": 42},
]

run_summaries = []

for i, params in enumerate(param_grid, start=1):
    with mlflow.start_run(run_name=f"rf_run_{i}"):
        # split data
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=params["random_state"]
        )

        # define model
        model = RandomForestRegressor(
            n_estimators=params["n_estimators"],
            max_depth=params["max_depth"],
            random_state=params["random_state"],
            n_jobs=-1
        )

        # train
        model.fit(X_train, y_train)

        # predict
        y_pred = model.predict(X_test)

        # metrics
        mae = mean_absolute_error(y_test, y_pred)
        mse = mean_squared_error(y_test, y_pred)
        rmse = np.sqrt(mse)
        r2 = r2_score(y_test, y_pred)

        # log params
        mlflow.log_param("model_type", "RandomForestRegressor")
        mlflow.log_param("n_estimators", params["n_estimators"])
        mlflow.log_param("max_depth", params["max_depth"])
        mlflow.log_param("random_state", params["random_state"])
        mlflow.log_param("features", ", ".join(feature_cols))

        # log metrics
        mlflow.log_metric("mae", mae)
        mlflow.log_metric("mse", mse)
        mlflow.log_metric("rmse", rmse)
        mlflow.log_metric("r2", r2)

        # log model
        mlflow.sklearn.log_model(model, "random_forest_model")

        # create and log artifacts
        with tempfile.TemporaryDirectory() as tmpdir:
            # Artifact 1: residual plot
            residual_plot_path = os.path.join(tmpdir, "residuals.png")
            residuals = y_test - y_pred

            plt.figure(figsize=(8, 5))
            plt.scatter(y_pred, residuals, alpha=0.5)
            plt.axhline(0, linestyle="--")
            plt.xlabel("Predicted Finish Position")
            plt.ylabel("Residuals")
            plt.title("Residual Plot")
            plt.tight_layout()
            plt.savefig(residual_plot_path)
            plt.close()

            mlflow.log_artifact(residual_plot_path, artifact_path=f"rf_run_{i}")

            # Artifact 2: feature importance CSV
            feature_importance_df = pd.DataFrame({
                "feature": feature_cols,
                "importance": model.feature_importances_
            }).sort_values("importance", ascending=False)

            feature_importance_path = os.path.join(tmpdir, "feature_importance.csv")
            feature_importance_df.to_csv(feature_importance_path, index=False)

            mlflow.log_artifact(feature_importance_path, artifact_path=f"rf_run_{i}")

        # save summary for notebook comparison
        run_summaries.append({
            "run_name": f"rf_run_{i}",
            "n_estimators": params["n_estimators"],
            "max_depth": params["max_depth"],
            "mae": mae,
            "mse": mse,
            "rmse": rmse,
            "r2": r2
        })

In [0]:
run_summary_df = pd.DataFrame(run_summaries).sort_values("rmse")
run_summary_df

In [0]:
best_run = run_summary_df.iloc[0]
best_run

## Best Model Selection

I conducted 12 experiments using different combinations of hyperparameters for the Random Forest model, varying `n_estimators` and `max_depth`.

To evaluate performance, I primarily used RMSE as the main metric, since it penalizes larger prediction errors more heavily. I also considered MAE and R² to ensure the model performs consistently across multiple metrics.

Among all runs, the best-performing model was **rf_run_3**, which used `n_estimators = 50` and `max_depth = 8`. This model achieved the lowest RMSE (approximately 3.376) and the lowest MAE among all runs, while also maintaining the highest R² value. 

This indicates that the model provides the most accurate and stable predictions of finishing position. Compared to deeper or more complex models, this configuration achieves a good balance between model complexity and generalization, avoiding overfitting while still capturing important patterns in the data.

Overall, rf_run_3 was selected as the best model because it consistently performed well across all evaluation metrics.